# Customer Churn Analysis

## 02. Data Cleaning

### Objective

The objective of this notebook is to identify and correct data quality issues before performing the exploratory analysis.

The cleaning process includes:

- Standardizing column names.
- Removing unnecessary whitespace.
- Checking duplicated records.
- Converting incorrect data types.
- Handling missing or invalid values.
- Validating categorical values.
- Creating a clean dataset for future analysis.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
RAW_DATA_PATH = Path(
    "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

PROCESSED_DATA_PATH = Path(
    "../data/processed/telco_customer_churn_clean.csv"
)

print("Raw dataset:", RAW_DATA_PATH.resolve())
print("Processed dataset:", PROCESSED_DATA_PATH.resolve())

Raw dataset: /home/sistemas-global-ice/Documents/projects/customer-churn-analysis/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
Processed dataset: /home/sistemas-global-ice/Documents/projects/customer-churn-analysis/data/processed/telco_customer_churn_clean.csv


## Load the Raw Dataset

The original dataset is loaded from the `data/raw` directory. A copy is created so the raw data remains unchanged.

In [4]:
DATA_PATH = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df_raw = pd.read_csv(DATA_PATH)

df = df_raw.copy()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 7,043
Columns: 21


In [5]:
df = df_raw.copy()

In [6]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Standardize Column Names

Column names are converted to lowercase and snake case to improve readability and maintain a consistent naming convention.

In [7]:
df.columns = (
    df.columns
    .str.strip()
    # KEEP LOWERCASE AND REPLACE SPACES WITH UNDERSCORES
    .str.replace("([a-z0-9])([A-Z])", r"\1_\2", regex=True)
    # PUT ALL IN LOWERCASE
    .str.lower()
)

df.columns.tolist()

['customer_id',
 'gender',
 'senior_citizen',
 'partner',
 'dependents',
 'tenure',
 'phone_service',
 'multiple_lines',
 'internet_service',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'contract',
 'paperless_billing',
 'payment_method',
 'monthly_charges',
 'total_charges',
 'churn']

In [8]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

text_columns.tolist()

['customer_id',
 'gender',
 'partner',
 'dependents',
 'phone_service',
 'multiple_lines',
 'internet_service',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'contract',
 'paperless_billing',
 'payment_method',
 'total_charges',
 'churn']

In [9]:
# Strip leading and trailing whitespace from text columns
for column in text_columns:
    df[column] = df[column].str.strip()

## Duplicate Records

Duplicate rows and duplicate customer identifiers are checked separately. Each customer should have a unique identifier.

In [10]:
duplicate_rows = df.duplicated().sum()
duplicate_customer_ids = df["customer_id"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate customer IDs:", duplicate_customer_ids)
print("Total records:", len(df))
print("Unique customers:", df["customer_id"].nunique())

Duplicate rows: 0
Duplicate customer IDs: 0
Total records: 7043
Unique customers: 7043


In [11]:
empty_strings = (df[text_columns] == "").sum()

empty_strings[empty_strings > 0]

total_charges    11
dtype: int64

## Missing Values in Total Charges

The `total_charges` column contains blank strings. These values were not initially detected as missing because they were stored as text.

Blank strings are converted to `NaN` before changing the column to a numeric data type.

In [12]:
df["total_charges"] = df["total_charges"].replace("", np.nan)
df["total_charges"].isna().sum()

np.int64(11)

In [13]:
# Convert the "total_charges" column to numeric, coercing errors to NaN
df["total_charges"] = pd.to_numeric(
    df["total_charges"],
    # coerce means that if there are any values that cannot be converted to numeric, they will be set to NaN instead of raising an error
    errors="coerce"
)
df["total_charges"].dtype

dtype('float64')

In [14]:
# Explore the rows with missing total charges
missing_total_charges = df[
    df["total_charges"].isna()
]

missing_total_charges[
    [
        "customer_id",
        "tenure",
        "monthly_charges",
        "total_charges",
        "contract",
        "churn"
    ]
]

,customer_id,tenure,monthly_charges,total_charges,contract,churn
488,4472-LVYGI,0,52.55,NaN,Two year,No
753,3115-CZMZD,0,20.25,NaN,Two year,No
936,5709-LVOEQ,0,80.85,NaN,Two year,No
1082,4367-NUYAO,0,25.75,NaN,Two year,No
1340,1371-DWPAZ,0,56.05,NaN,Two year,No
3331,7644-OMVMY,0,19.85,NaN,Two year,No
3826,3213-VVOLG,0,25.35,NaN,Two year,No
4380,2520-SGTTA,0,20.00,NaN,Two year,No
5218,2923-ARZLG,0,19.70,NaN,One year,No
6670,4075-WKNIU,0,73.35,NaN,Two year,No


In [15]:
# This means that there are 11 customers with missing total charges, and all of them have a tenure of 0 months. This is likely because they are new customers who have not yet been billed for any services.
missing_total_charges["tenure"].value_counts()

tenure
0    11
Name: count, dtype: int64

In [16]:
df.loc[
    df["total_charges"].isna() & (df["tenure"] == 0),
    "total_charges"
] = 0
df["total_charges"].isna().sum()

np.int64(0)

The missing `total_charges` values belong to customers with zero months of tenure. These records likely represent recently registered customers who have not accumulated charges yet.

Instead of removing these customers, their total charges were assigned a value of zero.


## Senior Citizen Variable

The `senior_citizen` column is stored as an integer, but it represents a categorical characteristic rather than a numerical measurement.

The values are converted to `Yes` and `No` to make the variable easier to interpret.

In [17]:
df["senior_citizen"] = df["senior_citizen"].map({
    0: "No",
    1: "Yes"
})
df["senior_citizen"].value_counts()

senior_citizen
No     5901
Yes    1142
Name: count, dtype: int64

In [18]:
df[
    [
        "tenure",
        "monthly_charges",
        "total_charges"
    ]
].describe()

numeric_columns = [
    "tenure",
    "monthly_charges",
    "total_charges"
]

for column in numeric_columns:
    negative_values = (df[column] < 0).sum()
    print(f"{column}: {negative_values} negative values")

tenure: 0 negative values
monthly_charges: 0 negative values
total_charges: 0 negative values


In [19]:
categorical_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


customer_id
customer_id
7590-VHVEG    1
5575-GNVDE    1
3668-QPYBK    1
7795-CFOCW    1
9237-HQITU    1
             ..
6840-RESVB    1
2234-XADUH    1
4801-JZAZL    1
8361-LTMKD    1
3186-AJIEK    1
Name: count, Length: 7043, dtype: int64

gender
gender
Male      3555
Female    3488
Name: count, dtype: int64

senior_citizen
senior_citizen
No     5901
Yes    1142
Name: count, dtype: int64

partner
partner
No     3641
Yes    3402
Name: count, dtype: int64

dependents
dependents
No     4933
Yes    2110
Name: count, dtype: int64

phone_service
phone_service
Yes    6361
No      682
Name: count, dtype: int64

multiple_lines
multiple_lines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

internet_service
internet_service
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

online_security
online_security
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype

In [20]:
cardinality = (
    df[categorical_columns]
    .nunique()
    .sort_values(ascending=False)
)

cardinality

cardinality_table = pd.DataFrame({
    "column": categorical_columns,
    "unique_values": [
        df[column].nunique()
        for column in categorical_columns
    ]
}).sort_values(
    by="unique_values",
    ascending=False
)

cardinality_table

,column,unique_values
0,customer_id,7043
16,payment_method,4
13,streaming_movies,3
9,online_backup,3
7,internet_service,3
8,online_security,3
11,tech_support,3
12,streaming_tv,3
10,device_protection,3
6,multiple_lines,3


In [21]:
df["churn"].value_counts(dropna=False)


churn
No     5174
Yes    1869
Name: count, dtype: int64

In [22]:
expected_churn_values = {"Yes", "No"}
actual_churn_values = set(df["churn"].dropna().unique())

unexpected_values = (
    actual_churn_values - expected_churn_values
)

print("Unexpected churn values:", unexpected_values)

Unexpected churn values: set()


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        7043 non-null   str    
 1   gender             7043 non-null   str    
 2   senior_citizen     7043 non-null   str    
 3   partner            7043 non-null   str    
 4   dependents         7043 non-null   str    
 5   tenure             7043 non-null   int64  
 6   phone_service      7043 non-null   str    
 7   multiple_lines     7043 non-null   str    
 8   internet_service   7043 non-null   str    
 9   online_security    7043 non-null   str    
 10  online_backup      7043 non-null   str    
 11  device_protection  7043 non-null   str    
 12  tech_support       7043 non-null   str    
 13  streaming_tv       7043 non-null   str    
 14  streaming_movies   7043 non-null   str    
 15  contract           7043 non-null   str    
 16  paperless_billing  7043 non-null   

In [24]:
quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(),
    "duplicate_values": [
        df[column].duplicated().sum()
        for column in df.columns
    ]
})

quality_report

,data_type,missing_values,unique_values,duplicate_values
customer_id,str,0,7043,0
gender,str,0,2,7041
senior_citizen,str,0,2,7041
partner,str,0,2,7041
dependents,str,0,2,7041
tenure,int64,0,73,6970
phone_service,str,0,2,7041
multiple_lines,str,0,3,7040
internet_service,str,0,3,7040
online_security,str,0,3,7040


In [25]:
assert df.shape[0] == 7043
assert df["customer_id"].is_unique
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0
assert (df["tenure"] >= 0).all()
assert (df["monthly_charges"] >= 0).all()
assert (df["total_charges"] >= 0).all()

print("All data quality checks passed.")

All data quality checks passed.


In [26]:
cleaning_summary = pd.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Missing values",
        "Duplicate rows"
    ],
    "raw_dataset": [
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw.isna().sum().sum(),
        df_raw.duplicated().sum()
    ],
    "clean_dataset": [
        df.shape[0],
        df.shape[1],
        df.isna().sum().sum(),
        df.duplicated().sum()
    ]
})

cleaning_summary

,metric,raw_dataset,clean_dataset
0,Rows,7043,7043
1,Columns,21,21
2,Missing values,0,0
3,Duplicate rows,0,0


## Export the Clean Dataset

The cleaned dataset is saved in the `data/processed` directory. This file will be used in the exploratory analysis and machine learning notebooks.

In [27]:
PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    PROCESSED_DATA_PATH,
    index=False
)

print(
    f"Clean dataset saved to: "
    f"{PROCESSED_DATA_PATH.resolve()}"
)

Clean dataset saved to: /home/sistemas-global-ice/Documents/projects/customer-churn-analysis/data/processed/telco_customer_churn_clean.csv


## Cleaning Summary

The following transformations were applied:

- Column names were standardized using snake case.
- Leading and trailing whitespace was removed from text variables.
- Duplicate records and customer identifiers were checked.
- Blank values in `total_charges` were identified.
- The `total_charges` column was converted from text to numeric.
- Missing total charges for customers with zero tenure were assigned a value of zero.
- The `senior_citizen` variable was converted into a readable categorical variable.
- Numerical values were checked for invalid negative values.
- Categorical variables were reviewed for inconsistent values.
- Data quality assertions were successfully completed.
- The cleaned dataset was exported to `data/processed`.

The resulting dataset contains 7,043 customers, 21 variables, no duplicated records, and no missing values.